requirements.txt
# 🛡️ LeakageGuard AI: Quant Code Auditor
**Veritas AI International High School Hackathon 2026**

**Track:** AI + Finance / Quantitative Development
**One-sentence pitch:** LeakageGuard AI helps quantitative developers and algo-traders catch hidden lookahead bias and time-series data leakage in their Python trading strategies before they deploy them to backtests or live markets.

**How to run:** Runtime → Run all. It takes under a minute and requires no API keys or local terminal setup.


## What goes in, and what comes out?
We pass a Python trading strategy script containing time-series operations into our Abstract Syntax Tree (AST) visitor. The model scans the structural code logic rather than doing naive text matching.


core/__init__.py

core/parser.py

In [ ]:
import ast

class CodeParser:
    """Parses target Python script files or raw code strings into an Abstract Syntax Tree (AST)."""

    @staticmethod
    def parse_file(file_path: str) -> ast.AST:
        with open(file_path, "r", encoding="utf-8") as file:
            source_code = file.read()
        return ast.parse(source_code)

    @staticmethod
    def parse_string(source_code: str) -> ast.AST:
        return ast.parse(source_code)

## 2. Where does it fail?
Static AST analysis cannot evaluate runtime values or dynamic variable indirection (e.g., if a developer stores a lookahead offset in an external config file or a dynamically computed variable rather than a raw constant like `-1` or `+1`).

core/rules.py

In [ ]:
import ast

class LookaheadVisitor(ast.NodeVisitor):
    """AST Visitor that scans syntax trees for lookahead bias and negative Pandas shifts."""
    def __init__(self):
        self.violations = []

    def visit_BinOp(self, node: ast.BinOp):
        if isinstance(node.op, ast.Add):
            if isinstance(node.right, ast.Constant) and isinstance(node.right.value, int) and node.right.value > 0:
                if isinstance(node.left, ast.Name):
                    self.violations.append({
                        "line": getattr(node, 'lineno', 'Unknown'),
                        "type": "Lookahead Indexing Bias",
                        "detail": f"Illegal future offset (+{node.right.value}) added to time index '{node.left.id}'."
                    })
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        if isinstance(node.func, ast.Attribute) and node.func.attr == "shift":
            for arg in node.args:
                if isinstance(arg, ast.UnaryOp) and isinstance(arg.op, ast.USub):
                    if isinstance(arg.operand, ast.Constant):
                        self.violations.append({
                            "line": getattr(node, 'lineno', 'Unknown'),
                            "type": "Pandas Data Leakage",
                            "detail": f"Negative shift shift(-{arg.operand.value}) references future prices in backtest."
                        })
        self.generic_visit(node)

core/auditor.py

In [ ]:
import ast

# --- 1. PARSER ---
class CodeParser:
    """Parses target Python script files or raw code strings into an Abstract Syntax Tree (AST)."""
    @staticmethod
    def parse_string(source_code: str) -> ast.AST:
        return ast.parse(source_code)

# --- 2. RULES & VISITOR ---
class LookaheadVisitor(ast.NodeVisitor):
    """AST Visitor that scans syntax trees for lookahead bias and negative Pandas shifts."""
    def __init__(self):
        self.violations = []

    def visit_BinOp(self, node: ast.BinOp):
        if isinstance(node.op, ast.Add):
            if isinstance(node.right, ast.Constant) and isinstance(node.right.value, int) and node.right.value > 0:
                if isinstance(node.left, ast.Name):
                    self.violations.append({
                        "line": getattr(node, 'lineno', 'Unknown'),
                        "type": "Lookahead Indexing Bias",
                        "detail": f"Illegal future offset (+{node.right.value}) added to time index '{node.left.id}'."
                    })
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        if isinstance(node.func, ast.Attribute) and node.func.attr == "shift":
            for arg in node.args:
                if isinstance(arg, ast.UnaryOp) and isinstance(arg.op, ast.USub):
                    if isinstance(arg.operand, ast.Constant):
                        self.violations.append({
                            "line": getattr(node, 'lineno', 'Unknown'),
                            "type": "Pandas Data Leakage",
                            "detail": f"Negative shift shift(-{arg.operand.value}) references future prices in backtest."
                        })
        self.generic_visit(node)

# --- 3. AUDITOR ---
class QuantAuditor:
    """Orchestrates static analysis audits on quantitative strategy code."""

    def __init__(self):
        self.parser = CodeParser()

    def audit_string(self, source_code: str) -> dict:
        try:
            tree = self.parser.parse_string(source_code)
            return self._evaluate_ast(tree)
        except SyntaxError as e:
            return {"status": "SYNTAX_ERROR", "errors": [{"line": e.lineno, "type": "SyntaxError", "detail": str(e)}]}

    def _evaluate_ast(self, tree: ast.AST) -> dict:
        visitor = LookaheadVisitor()
        visitor.visit(tree)
        if visitor.violations:
            return {"status": "FAILED", "errors": visitor.violations}
        return {"status": "PASSED", "errors": []}

print("✅ QuantAuditor and core engine loaded successfully in Colab!")

✅ QuantAuditor and core engine loaded successfully in Colab!


main.py



In [ ]:
# Test your auditing engine directly inside Colab
auditor = QuantAuditor()

# Sample strategy code with intentional lookahead bias
sample_strategy = """
import pandas as pd
def run_strategy(df):
    # Intentional lookahead bias / data leakage
    df['future_price'] = df['close'].shift(-1)
    df['adjusted'] = df['close'] + 10
    return df
"""

print("[*] Running LeakageGuard AI audit on strategy...\n")
result = auditor.audit_string(sample_strategy)

if result["status"] == "PASSED":
    print("✅ PASSED: No lookahead bias or data leakage detechttps://calendar.notion.com/2026/10/30$0ted!")
elif result["status"] == "FAILED":
    print("❌ FAILED: Security violations found:")
    for err in result["errors"]:
        print(f"  - [Line {err['line']}] [{err['type']}]: {err['detail']}")

[*] Running LeakageGuard AI audit on strategy...

❌ FAILED: Security violations found:
  - [Line 5] [Pandas Data Leakage]: Negative shift shift(-1) references future prices in backtest.


app.py

In [ ]:
# --- COLAB DEMO TEST CELL ---
# This tests your QuantAuditor engine directly inside your notebook

auditor = QuantAuditor()

# Sample strategy with lookahead bias
strategy_code = """
import pandas as pd

def trading_strategy(df):
    # Intentional lookahead bias
    df['future_price'] = df['close'].shift(-1)

    for t in range(len(df)-1):
        if df['close'].iloc[t + 1] > df['close'].iloc[t]:
            print("BUY")
"""

print("[*] Running LeakageGuard AI audit...\n")
result = auditor.audit_string(strategy_code)

if result["status"] == "PASSED":
    print("✅ PASSED: No lookahead bias or time-series data leakage detected!")
elif result["status"] == "FAILED":
    print("❌ AUDIT FAILED: Lookahead Violations Detected!")
    for idx, err in enumerate(result["errors"], 1):
        print(f"  - Violation #{idx} (Line {err['line']}) [{err['type']}]: {err['detail']}")

[*] Running LeakageGuard AI audit...

❌ AUDIT FAILED: Lookahead Violations Detected!
  - Violation #1 (Line 6) [Pandas Data Leakage]: Negative shift shift(-1) references future prices in backtest.
  - Violation #2 (Line 9) [Lookahead Indexing Bias]: Illegal future offset (+1) added to time index 't'.


examples/flawed_strategy.py

In [ ]:
import pandas as pd

def generate_signals(df):
    df['future_close'] = df['close'].shift(-1)
    if df['close'].iloc[t + 1] > df['close'].iloc[t]:
        return "BUY"

examples/clean_strategy.py

In [ ]:
import pandas as pd

def generate_signals(df):
    df['sma_20'] = df['close'].rolling(window=20).mean()
    return df['sma_20']

tests/test_auditor.py

In [ ]:
import unittest

class TestQuantAuditor(unittest.TestCase):
    def test_flawed_code_fails(self):
        auditor = QuantAuditor()
        result = auditor.audit_string("future = close.shift(-1)\nval = close[t + 1]")
        self.assertEqual(result["status"], "FAILED")

    def test_clean_code_passes(self):
        auditor = QuantAuditor()
        result = auditor.audit_string("sma = close.rolling(20).mean()")
        self.assertEqual(result["status"], "PASSED")

# Run the unit tests safely inside Google Colab
if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


## Sources & References
* **Python Software Foundation:** `ast` — Abstract Syntax Tree documentation, Python Standard Library. https://docs.python.org/3/library/ast.html
* **Pandas Development Team:** Pandas Documentation — Time Series / Date functionality (`DataFrame.shift`, rolling windows). https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html
* **Quantitative Finance Methodology:** López de Prado, Marcos. *Advances in Financial Machine Learning*. McGraw-Hill Education, 2018. (Referenced for identifying data leakage and lookahead bias in financial backtesting).
* **Tooling:** Built using Python, Pandas, and Streamlit.